# Heart Disease Prediction: Exploratory Data Analysis & Modeling

Notebook ini bertujuan untuk melakukan eksplorasi data (EDA), pra-pemrosesan data, dan melatih model **Random Forest Classifier** untuk memprediksi risiko penyakit jantung.
Langkah-langkah yang akan kita lakukan mengikuti *best practices* Machine Learning, termasuk memisahkan data (split) sebelum melakukan transformasi/encoding untuk mencegah kebocoran data (*data leakage*).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

: 

## 1. Memuat Data
Kita memuat dataset dari folder `data` yang berada satu tingkat di atas direktori ini.

In [ ]:
# Load data
df = pd.read_csv('../data/heart.csv')
df.head()

## 2. Eksplorasi Data (EDA)
Melihat info dataset, nilai kosong (missing values), dan persebaran target `HeartDisease`.

In [ ]:
print("\n--- Info Dataset ---")
df.info()

print("\n--- Pengecekan Missing Values ---")
print(df.isnull().sum())

print("\n--- Distribusi Target ---")
print(df['HeartDisease'].value_counts())

# Visualisasi target
sns.countplot(x='HeartDisease', data=df)
plt.title('Distribusi Penyakit Jantung (0: Negatif, 1: Positif)')
plt.show()

## 3. Data Splitting
Sangat penting untuk memisahkan data *Training* dan *Test* terlebih dahulu sebelum melakukan preprocessing (seperti scaling dan encoding) agar model tidak tanpa sengaja "melihat" informasi dari data test.

In [ ]:
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# 80% data latih, 20% data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test:", X_test.shape)

## 4. Preprocessing
Data kita terdiri dari numerik dan kategorikal. 
- Numerik akan di-*scale* menggunakan `StandardScaler`.
- Kategorikal akan di-*encode* menggunakan `OneHotEncoder`.

In [ ]:
# Tentukan fitur kategorikal dan numerikal berdasarkan tipe data
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_features = X_train.select_dtypes(exclude=['object']).columns.tolist()

print("Fitur Numerikal:", numeric_features)
print("Fitur Kategorikal:", categorical_features)

# Buat transformer untuk preprocessing
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Gabungkan dengan ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

## 5. Membuat Pipeline Model
Kita menyatukan langkah `preprocessor` dan pelatihan model `RandomForestClassifier` menjadi satu alur utuh.

In [ ]:
# Inisialisasi model Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Buat Pipeline
clf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', rf_model)])

# Latih model hanya menggunakan data latih
clf_pipeline.fit(X_train, y_train)
print("Model berhasil dilatih!")

## 6. Evaluasi Model
Kita melihat seberapa baik model memprediksi data uji (`X_test`).

In [ ]:
# Prediksi
y_pred = clf_pipeline.predict(X_test)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix')
plt.show()

## 7. Menyimpan Model
Agar model bisa dipakai pada aplikasi web FastAPI (Backend), kita menyimpan keseluruhan `Pipeline` ke dalam file `.pkl`. Menyimpan Pipeline memastikan data input baru nantinya otomatis melalui proses preprocessing (scaling & encoding) yang sama.

In [ ]:
import os

backend_dir = '../backend'
if not os.path.exists(backend_dir):
    os.makedirs(backend_dir)

model_path = os.path.join(backend_dir, 'model.pkl')
joblib.dump(clf_pipeline, model_path)

print(f"Model berhasil disimpan di: {model_path}")